<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Bayesian-Embeddings/blob/main/Sim1_Bayesian_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# SIMULATION 1 — JRSSB/JASA FINAL ROUNDED-PARAMETER DESIGN
# ============================================================
#
# Statistical objective:
#   1. Structural allocation recovery.
#   2. Posterior spectral/tail estimation.
#   3. Adaptive budget selection.
#   4. Predictive efficiency and parameter reduction.
#
# Design:
#   spectral regime = {clear, weak}
#   n_train         = {250, 500, 1000}
#
# IMPORTANT:
#   Final rounded-parameter design. Structural dimensions and
#   category cardinalities are retained from the successful pilot;
#   freely chosen numerical and computational constants are rounded.
#
# MCMC modifications:
#   1. Separate Gamma and mu Metropolis blocks.
#   2. Initial Gamma/mu proposal scales proportional to n^{-1/2}.
#   3. More responsive adaptation every 25 iterations.
#   4. Separate acceptance targets for Theta, Gamma and mu.
#   5. Adaptation stops before burn-in ends.
#   6. Acceptance diagnostics are computed POST BURN-IN.
#
# Bayesian stage:
#   fixed-dimensional singular-value shrinkage prior
#   +
#   blocked Metropolis-within-Gibbs MCMC.
#
# Dynamic programming:
#   exact Multiple-Choice Knapsack DP.
#
# GOOGLE COLAB — ONE CELL
#
# At startup:
#   1 = start from beginning
#   2 = resume from latest compatible checkpoint
#
# Checkpoints:
#   - after every completed Monte Carlo replication
#   - additionally approximately every 20 minutes
# ============================================================


# ============================================================
# 0. START / RESUME
# ============================================================

MODE = input(
    "Simulation 1 rounded-final: type 1 to START FROM THE BEGINNING "
    "or 2 to RESUME FROM THE LAST CHECKPOINT: "
).strip()

if MODE not in {"1", "2"}:
    raise ValueError(
        "Please run the cell again and enter either 1 or 2."
    )


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import gc
import json
import math
import time
import pickle
import random
import hashlib
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

DEVICE = torch.device("cpu")

torch.set_num_threads(
    max(
        1,
        min(
            4,
            os.cpu_count() or 1
        )
    )
)


# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

try:

    from google.colab import drive

    drive.mount(
        "/content/drive"
    )

    ROOT = (
        "/content/drive/MyDrive/"
        "JRSSB_Embedding_Allocation/"
        "Simulation1_rounded_final"
    )

except Exception:

    ROOT = (
        "./JRSSB_Embedding_Allocation/"
        "Simulation1_rounded_final"
    )

    print(
        "Google Drive was not detected. "
        "Using local directory:",
        ROOT
    )

os.makedirs(
    ROOT,
    exist_ok=True
)

CHECKPOINT_PATH = os.path.join(
    ROOT,
    "simulation1_rounded_final_checkpoint.pkl"
)

RESULTS_PATH = os.path.join(
    ROOT,
    "simulation1_rounded_final_replication_results.csv"
)

BUDGET_RESULTS_PATH = os.path.join(
    ROOT,
    "simulation1_rounded_final_budget_results.csv"
)

SUMMARY_PATH = os.path.join(
    ROOT,
    "simulation1_rounded_final_summary.csv"
)


# ============================================================
# 3. SIMULATION CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# False runs the final 100-replication experiment.
# True can be used for a shorter diagnostic run.
# ------------------------------------------------------------

PILOT_MODE = False

BASE_SEED = 271828

CHECKPOINT_SECONDS = 20 * 60


# ------------------------------------------------------------
# Categorical predictors
# ------------------------------------------------------------

N_LEVELS = [
    8,
    10,
    12,
    9
]

M = len(
    N_LEVELS
)


# ------------------------------------------------------------
# Latent and response dimensions
# ------------------------------------------------------------

P_Z = 4

Q = 4

P_U = 2

K_LIST = [
    min(
        nj - 1,
        P_Z
    )
    for nj in N_LEVELS
]


# ------------------------------------------------------------
# Validation / testing
# ------------------------------------------------------------

N_VAL = 500

N_TEST = 3000


# ------------------------------------------------------------
# Adaptive-budget tolerance
# ------------------------------------------------------------

ZETA_N = 0.0015


# ------------------------------------------------------------
# Bayesian prior
# ------------------------------------------------------------

THETA_LAMBDA0 = 0.05

TAU_GAMMA = 2.5

TAU_MU = 2.5

SIGMA_A0 = 2.5

SIGMA_B0 = 0.02


# ------------------------------------------------------------
# Reduced MLP
# ------------------------------------------------------------

RED_HIDDEN = (
    15,
    10
)

RED_WEIGHT_DECAY = 5e-4


# ------------------------------------------------------------
# Training / MCMC settings
# ------------------------------------------------------------

if PILOT_MODE:

    N_REPS = 10

    REF_EPOCHS = 300

    RED_EPOCHS = 250

    REF_RESTARTS = 5

    RED_RESTARTS = 5

    MCMC_ITERS = 750

    MCMC_BURN = 375

    MCMC_THIN = 5

else:

    # Final Monte Carlo experiment.
    N_REPS = 100

    REF_EPOCHS = 400

    RED_EPOCHS = 300

    REF_RESTARTS = 5

    RED_RESTARTS = 5

    MCMC_ITERS = 1500

    MCMC_BURN = 750

    MCMC_THIN = 5


# ------------------------------------------------------------
# MCMC tuning constants
# ------------------------------------------------------------

MCMC_ADAPT_EVERY = 25

MCMC_ADAPT_FRACTION = 0.80

MCMC_TARGET_THETA = 0.25

MCMC_TARGET_GAMMA = 0.25

MCMC_TARGET_MU = 0.30

# Base proposal scales are calibrated at n = 250.
# Gamma and mu are automatically rescaled by sqrt(250/n).

THETA_INITIAL_STEP = 0.005

GAMMA_INITIAL_STEP = 0.010

MU_INITIAL_STEP = 0.010

MCMC_REFERENCE_N = 250.0

# Robbins-Monro style multiplicative adaptation intensity.

MCMC_ADAPT_GAIN = 0.80

THETA_STEP_MIN = 0.0005

THETA_STEP_MAX = 0.050

GAMMA_STEP_MIN = 0.0005

GAMMA_STEP_MAX = 0.050

MU_STEP_MIN = 0.0005

MU_STEP_MAX = 0.050


# ------------------------------------------------------------
# Clean factorial design
# ------------------------------------------------------------

NOISE_SD = 0.10

SCENARIOS = []

for spectral_regime in [
    "clear",
    "weak"
]:

    for n_train in [
        250,
        500,
        1000
    ]:

        SCENARIOS.append(
            {
                "name":
                    f"{spectral_regime}_n{n_train}",

                "n_train":
                    n_train,

                "spectral_regime":
                    spectral_regime,

                "noise_sd":
                    NOISE_SD
            }
        )


# ============================================================
# 4. BUDGET GRID
# ============================================================

MIN_COST = sum(
    N_LEVELS
)

MAX_COST = sum(
    N_LEVELS[j]
    * K_LIST[j]
    for j in range(M)
)

BUDGETS = [
    MIN_COST,
    60,
    80,
    100,
    120,
    140,
    MAX_COST
]


# ============================================================
# 5. CONFIGURATION HASH
# ============================================================

CONFIG_FOR_HASH = {

    "pilot":
        PILOT_MODE,

    "n_reps":
        N_REPS,

    "n_levels":
        N_LEVELS,

    "p_z":
        P_Z,

    "q":
        Q,

    "p_u":
        P_U,

    "n_val":
        N_VAL,

    "n_test":
        N_TEST,

    "noise_sd":
        NOISE_SD,

    "budgets":
        BUDGETS,

    "zeta_n":
        ZETA_N,

    "theta_lambda0":
        THETA_LAMBDA0,

    "tau_gamma":
        TAU_GAMMA,

    "tau_mu":
        TAU_MU,

    "sigma_a0":
        SIGMA_A0,

    "sigma_b0":
        SIGMA_B0,

    "red_hidden":
        RED_HIDDEN,

    "red_weight_decay":
        RED_WEIGHT_DECAY,

    "ref_epochs":
        REF_EPOCHS,

    "red_epochs":
        RED_EPOCHS,

    "ref_restarts":
        REF_RESTARTS,

    "red_restarts":
        RED_RESTARTS,

    "mcmc_iters":
        MCMC_ITERS,

    "mcmc_burn":
        MCMC_BURN,

    "mcmc_thin":
        MCMC_THIN,

    "adapt_every":
        MCMC_ADAPT_EVERY,

    "adapt_fraction":
        MCMC_ADAPT_FRACTION,

    "target_theta":
        MCMC_TARGET_THETA,

    "target_gamma":
        MCMC_TARGET_GAMMA,

    "target_mu":
        MCMC_TARGET_MU,

    "theta_initial_step":
        THETA_INITIAL_STEP,

    "gamma_initial_step":
        GAMMA_INITIAL_STEP,

    "mu_initial_step":
        MU_INITIAL_STEP,

    "adapt_gain":
        MCMC_ADAPT_GAIN,

    "theta_step_min":
        THETA_STEP_MIN,

    "theta_step_max":
        THETA_STEP_MAX,

    "gamma_step_min":
        GAMMA_STEP_MIN,

    "gamma_step_max":
        GAMMA_STEP_MAX,

    "mu_step_min":
        MU_STEP_MIN,

    "mu_step_max":
        MU_STEP_MAX,

    "scenarios":
        SCENARIOS
}

CONFIG_SIGNATURE = hashlib.md5(
    json.dumps(
        CONFIG_FOR_HASH,
        sort_keys=True
    ).encode()
).hexdigest()


# ============================================================
# 6. REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)


# ============================================================
# 7. BASIC LINEAR-ALGEBRA HELPERS
# ============================================================

def random_orthogonal(
    d,
    rng
):

    A = rng.normal(
        size=(
            d,
            d
        )
    )

    Qm, R = np.linalg.qr(
        A
    )

    signs = np.sign(
        np.diag(
            R
        )
    )

    signs[
        signs == 0
    ] = 1.0

    Qm = (
        Qm
        * signs
    )

    return Qm


def centered_orthonormal_matrix(
    n,
    k,
    rng
):

    A = rng.normal(
        size=(
            n,
            k
        )
    )

    A -= A.mean(
        axis=0,
        keepdims=True
    )

    Qm, _ = np.linalg.qr(
        A
    )

    Qm -= Qm.mean(
        axis=0,
        keepdims=True
    )

    Qm, _ = np.linalg.qr(
        Qm
    )

    return Qm[
        :,
        :k
    ]


def make_theta_from_singular_values(
    n,
    pz,
    singular_values,
    rng
):

    k = min(
        n - 1,
        pz
    )

    U = centered_orthonormal_matrix(
        n,
        k,
        rng
    )

    V = random_orthogonal(
        pz,
        rng
    )[
        :,
        :k
    ]

    s = np.asarray(
        singular_values[
            :k
        ],
        dtype=float
    )

    theta = (
        U
        @ np.diag(
            s
        )
        @ V.T
    )

    theta -= theta.mean(
        axis=0,
        keepdims=True
    )

    return theta


# ============================================================
# 8. TRUE SPECTRAL STRUCTURE
# ============================================================

def get_spectral_values(
    regime
):

    if regime == "clear":

        spectra = [

            [
                1.80,
                0.30,
                0.10,
                0.05
            ],

            [
                1.60,
                0.70,
                0.20,
                0.05
            ],

            [
                2.00,
                1.00,
                0.40,
                0.10
            ],

            [
                1.50,
                0.50,
                0.20,
                0.05
            ]
        ]

    elif regime == "weak":

        spectra = [

            [
                1.80,
                1.20,
                0.90,
                0.60
            ],

            [
                1.60,
                1.30,
                1.00,
                0.70
            ],

            [
                2.00,
                1.60,
                1.30,
                1.00
            ],

            [
                1.50,
                1.10,
                0.85,
                0.60
            ]
        ]

    else:

        raise ValueError(
            "Unknown spectral regime."
        )

    return [
        np.asarray(
            spectra[j][
                :K_LIST[j]
            ],
            dtype=float
        )
        for j in range(M)
    ]


# ============================================================
# 9. TRUE INJECTIVE REFERENCE MLP
# ============================================================

def make_teacher_weights(
    seed=8877
):

    rng = np.random.default_rng(
        seed
    )

    W1 = random_orthogonal(
        P_Z,
        rng
    )

    W2 = random_orthogonal(
        P_Z,
        rng
    )

    b1 = np.array(
        [
            0.10,
            -0.10,
            0.05,
            -0.05
        ],
        dtype=float
    )

    b2 = np.array(
        [
            0.05,
            -0.05,
            0.05,
            -0.05
        ],
        dtype=float
    )

    return [
        (
            W1,
            b1
        ),
        (
            W2,
            b2
        )
    ]


TEACHER_WEIGHTS = (
    make_teacher_weights()
)


def mlp_forward_numpy(
    z,
    weights
):

    W1, b1 = weights[
        0
    ]

    W2, b2 = weights[
        1
    ]

    h = np.tanh(
        z
        @ W1.T
        +
        b1
    )

    return (
        h
        @ W2.T
        +
        b2
    )


# ============================================================
# 10. TRUE PARAMETERS
# ============================================================

def build_true_parameters(
    spectral_regime
):

    rng = np.random.default_rng(
        12345
    )

    singular_values = (
        get_spectral_values(
            spectral_regime
        )
    )

    theta_true = []

    for j in range(M):

        theta_true.append(
            make_theta_from_singular_values(
                N_LEVELS[j],
                P_Z,
                singular_values[j],
                rng
            )
        )

    gamma_true = rng.normal(
        scale=0.20,
        size=(
            P_U,
            P_Z
        )
    )

    mu_true = np.array(
        [
            0.10,
            -0.10,
            0.05,
            -0.05
        ]
    )

    return {

        "theta":
            theta_true,

        "gamma":
            gamma_true,

        "mu":
            mu_true,

        "singular_values":
            singular_values
    }


# ============================================================
# 11. DATA GENERATION
# ============================================================

def generate_dataset(
    n,
    true_par,
    noise_sd,
    seed
):

    rng = np.random.default_rng(
        seed
    )

    C = [

        rng.integers(
            0,
            N_LEVELS[j],
            size=n
        )

        for j in range(M)
    ]

    U = rng.normal(
        size=(
            n,
            P_U
        )
    )

    z = np.tile(
        true_par[
            "mu"
        ],
        (
            n,
            1
        )
    )

    z += (
        U
        @ true_par[
            "gamma"
        ]
    )

    for j in range(M):

        z += (
            true_par[
                "theta"
            ][j][
                C[j],
                :
            ]
        )

    y_mean = mlp_forward_numpy(
        z,
        TEACHER_WEIGHTS
    )

    y = (
        y_mean
        +
        noise_sd
        * rng.normal(
            size=y_mean.shape
        )
    )

    return (
        C,
        U,
        y,
        y_mean
    )


# ============================================================
# 12. REFERENCE MLP
# ============================================================

class ReferenceModel(
    nn.Module
):

    def __init__(
        self,
        seed=0
    ):

        super().__init__()

        set_seed(
            seed
        )

        self.embeddings = (
            nn.ModuleList(
                [
                    nn.Embedding(
                        N_LEVELS[j],
                        P_Z
                    )
                    for j in range(M)
                ]
            )
        )

        for emb in self.embeddings:

            nn.init.normal_(
                emb.weight,
                std=0.10
            )

        self.gamma = nn.Parameter(
            0.10
            * torch.randn(
                P_U,
                P_Z
            )
        )

        self.mu = nn.Parameter(
            torch.zeros(
                P_Z
            )
        )

        self.lin1 = nn.Linear(
            P_Z,
            P_Z
        )

        self.lin2 = nn.Linear(
            P_Z,
            P_Z
        )

        with torch.no_grad():

            q1, _ = torch.linalg.qr(
                torch.randn(
                    P_Z,
                    P_Z
                )
            )

            q2, _ = torch.linalg.qr(
                torch.randn(
                    P_Z,
                    P_Z
                )
            )

            self.lin1.weight.copy_(
                q1
            )

            self.lin2.weight.copy_(
                q2
            )

            self.lin1.bias.zero_()

            self.lin2.bias.zero_()

    def latent(
        self,
        C,
        U
    ):

        z = (
            self.mu.unsqueeze(
                0
            )
            +
            U
            @ self.gamma
        )

        for j in range(M):

            z = (
                z
                +
                self.embeddings[j](
                    C[j]
                )
            )

        return z

    def phi(
        self,
        z
    ):

        return self.lin2(
            torch.tanh(
                self.lin1(
                    z
                )
            )
        )

    def forward(
        self,
        C,
        U
    ):

        return self.phi(
            self.latent(
                C,
                U
            )
        )

    @torch.no_grad()
    def center_embeddings_(
        self
    ):

        for emb in self.embeddings:

            emb.weight -= (
                emb.weight.mean(
                    dim=0,
                    keepdim=True
                )
            )

    @torch.no_grad()
    def project_mlp_orthogonal_(
        self
    ):

        for layer in [
            self.lin1,
            self.lin2
        ]:

            U, _, Vh = (
                torch.linalg.svd(
                    layer.weight,
                    full_matrices=False
                )
            )

            layer.weight.copy_(
                U
                @ Vh
            )


# ============================================================
# 13. REFERENCE MODEL TRAINING
# ============================================================

def train_reference_once(
    C,
    U,
    y,
    seed,
    heartbeat=None
):

    set_seed(
        seed
    )

    model = ReferenceModel(
        seed=seed
    ).to(
        DEVICE
    )

    C_t = [
        torch.tensor(
            x,
            dtype=torch.long,
            device=DEVICE
        )
        for x in C
    ]

    U_t = torch.tensor(
        U,
        dtype=torch.float32,
        device=DEVICE
    )

    y_t = torch.tensor(
        y,
        dtype=torch.float32,
        device=DEVICE
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.005,
        weight_decay=5e-5
    )

    for epoch in range(
        REF_EPOCHS
    ):

        optimizer.zero_grad()

        pred = model(
            C_t,
            U_t
        )

        loss = F.mse_loss(
            pred,
            y_t
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0
        )

        optimizer.step()

        model.center_embeddings_()

        model.project_mlp_orthogonal_()

        if (
            heartbeat is not None
            and
            (epoch + 1) % 50 == 0
        ):

            heartbeat()

    with torch.no_grad():

        final_loss = float(
            F.mse_loss(
                model(
                    C_t,
                    U_t
                ),
                y_t
            ).cpu()
        )

    return (
        model,
        final_loss
    )


def train_reference_model(
    C,
    U,
    y,
    seed,
    heartbeat=None
):

    best_model = None

    best_loss = np.inf

    for r in range(
        REF_RESTARTS
    ):

        model, loss = (
            train_reference_once(
                C,
                U,
                y,
                seed
                +
                1000
                * r,
                heartbeat
            )
        )

        if loss < best_loss:

            if best_model is not None:

                del best_model

            best_model = model

            best_loss = loss

        else:

            del model

        gc.collect()

    return (
        best_model,
        best_loss
    )


@torch.no_grad()
def predict_reference(
    model,
    C,
    U
):

    C_t = [
        torch.tensor(
            x,
            dtype=torch.long,
            device=DEVICE
        )
        for x in C
    ]

    U_t = torch.tensor(
        U,
        dtype=torch.float32,
        device=DEVICE
    )

    return (
        model(
            C_t,
            U_t
        )
        .cpu()
        .numpy()
    )


def extract_reference_parameters(
    model
):

    theta = [
        emb.weight
        .detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
        for emb in model.embeddings
    ]

    theta = [
        x
        -
        x.mean(
            axis=0,
            keepdims=True
        )
        for x in theta
    ]

    gamma = (
        model.gamma
        .detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
    )

    mu = (
        model.mu
        .detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
    )

    W1 = (
        model.lin1.weight
        .detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
    )

    b1 = (
        model.lin1.bias
        .detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
    )

    W2 = (
        model.lin2.weight
        .detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
    )

    b2 = (
        model.lin2.bias
        .detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
    )

    weights = [
        (
            W1,
            b1
        ),
        (
            W2,
            b2
        )
    ]

    return (
        theta,
        gamma,
        mu,
        weights
    )


# ============================================================
# 14. FIXED-DIMENSION SPECTRAL SHRINKAGE PRIOR
# ============================================================

def log_theta_prior(
    theta
):

    k = min(
        theta.shape[0] - 1,
        theta.shape[1]
    )

    s = np.linalg.svd(
        theta,
        compute_uv=False
    )[
        :k
    ]

    lambdas = (
        THETA_LAMBDA0
        *
        np.arange(
            1,
            k + 1,
            dtype=float
        ) ** 2
    )

    return float(
        -0.5
        * np.sum(
            lambdas
            * s ** 2
        )
    )


def log_gamma_prior(
    gamma
):

    return float(
        -0.5
        * np.sum(
            gamma ** 2
        )
        / TAU_GAMMA ** 2
    )


def log_mu_prior(
    mu
):

    return float(
        -0.5
        * np.sum(
            mu ** 2
        )
        / TAU_MU ** 2
    )


# ============================================================
# 15. LIKELIHOOD
# ============================================================

def gaussian_loglik(
    y,
    yhat,
    sigma2
):

    residual = (
        y
        -
        yhat
    )

    return float(
        -0.5
        * y.size
        * np.log(
            2.0
            * np.pi
            * sigma2
        )
        -
        0.5
        * np.sum(
            residual ** 2
        )
        / sigma2
    )


def latent_numpy(
    C,
    U,
    theta,
    gamma,
    mu
):

    n = len(
        C[
            0
        ]
    )

    z = np.tile(
        mu,
        (
            n,
            1
        )
    )

    z += (
        U
        @ gamma
    )

    for j in range(M):

        z += (
            theta[j][
                C[j],
                :
            ]
        )

    return z


# ============================================================
# 16. TUNED BLOCKED MCMC
# ============================================================

def blocked_mcmc(
    C,
    U,
    y,
    theta_init,
    gamma_init,
    mu_init,
    fixed_mlp_weights,
    sigma2_init,
    seed,
    heartbeat=None
):

    rng = np.random.default_rng(
        seed
    )

    theta = [
        x.copy()
        for x in theta_init
    ]

    gamma = (
        gamma_init.copy()
    )

    mu = (
        mu_init.copy()
    )

    sigma2 = float(
        max(
            sigma2_init,
            1e-6
        )
    )


    # --------------------------------------------------------
    # Current latent representation and likelihood
    # --------------------------------------------------------

    z_cur = latent_numpy(
        C,
        U,
        theta,
        gamma,
        mu
    )

    yhat_cur = mlp_forward_numpy(
        z_cur,
        fixed_mlp_weights
    )

    ll_cur = gaussian_loglik(
        y,
        yhat_cur,
        sigma2
    )


    # --------------------------------------------------------
    # Current prior terms
    # --------------------------------------------------------

    theta_prior_cur = [
        log_theta_prior(
            x
        )
        for x in theta
    ]

    gamma_prior_cur = (
        log_gamma_prior(
            gamma
        )
    )

    mu_prior_cur = (
        log_mu_prior(
            mu
        )
    )


    # --------------------------------------------------------
    # Proposal scales
    #
    # Theta uses the same initial scale as the successful pilot.
    #
    # Gamma and mu use n^{-1/2} initialization. As n increases
    # the likelihood becomes more concentrated, so a fixed
    # random-walk scale would otherwise lead to decreasing
    # acceptance rates.
    # --------------------------------------------------------

    n_obs = len(
        C[
            0
        ]
    )

    n_scale = np.sqrt(
        MCMC_REFERENCE_N
        /
        float(
            n_obs
        )
    )

    theta_steps = np.full(
        M,
        THETA_INITIAL_STEP,
        dtype=float
    )

    gamma_step = float(
        GAMMA_INITIAL_STEP
        * n_scale
    )

    mu_step = float(
        MU_INITIAL_STEP
        * n_scale
    )


    # --------------------------------------------------------
    # Adaptation counters
    # --------------------------------------------------------

    theta_window_accept = np.zeros(
        M
    )

    theta_window_trials = np.zeros(
        M
    )

    gamma_window_accept = 0

    gamma_window_trials = 0

    mu_window_accept = 0

    mu_window_trials = 0


    # --------------------------------------------------------
    # Post-burn-in acceptance counters
    #
    # These, rather than acceptance during adaptation, are
    # reported as the MCMC diagnostic.
    # --------------------------------------------------------

    theta_post_accept = np.zeros(
        M
    )

    theta_post_trials = np.zeros(
        M
    )

    gamma_post_accept = 0

    gamma_post_trials = 0

    mu_post_accept = 0

    mu_post_trials = 0


    # --------------------------------------------------------
    # Posterior storage
    # --------------------------------------------------------

    singular_draws = [
        []
        for _ in range(M)
    ]

    sigma_draws = []


    # Stop adaptation before the end of burn-in.
    # The remaining burn-in iterations use a fixed kernel.

    adapt_stop = int(
        MCMC_ADAPT_FRACTION
        *
        MCMC_BURN
    )

    adaptation_round = 0


    for it in range(
        MCMC_ITERS
    ):


        # ====================================================
        # A. Predictor-specific Theta updates
        # ====================================================

        for j in range(M):

            proposal = (
                theta[j]
                +
                theta_steps[j]
                * rng.normal(
                    size=theta[j].shape
                )
            )

            proposal -= proposal.mean(
                axis=0,
                keepdims=True
            )

            delta_latent = (
                proposal[
                    C[j],
                    :
                ]
                -
                theta[j][
                    C[j],
                    :
                ]
            )

            z_prop = (
                z_cur
                +
                delta_latent
            )

            yhat_prop = mlp_forward_numpy(
                z_prop,
                fixed_mlp_weights
            )

            ll_prop = gaussian_loglik(
                y,
                yhat_prop,
                sigma2
            )

            prior_prop = log_theta_prior(
                proposal
            )

            log_ratio = (
                ll_prop
                +
                prior_prop
                -
                ll_cur
                -
                theta_prior_cur[j]
            )

            accepted = (
                np.log(
                    rng.uniform()
                )
                <
                log_ratio
            )

            if it < adapt_stop:

                theta_window_trials[j] += 1

            if it >= MCMC_BURN:

                theta_post_trials[j] += 1

            if accepted:

                theta[j] = proposal

                z_cur = z_prop

                yhat_cur = yhat_prop

                ll_cur = ll_prop

                theta_prior_cur[j] = (
                    prior_prop
                )

                if it < adapt_stop:

                    theta_window_accept[j] += 1

                if it >= MCMC_BURN:

                    theta_post_accept[j] += 1


        # ====================================================
        # B. Gamma block
        # ====================================================

        gamma_prop = (
            gamma
            +
            gamma_step
            * rng.normal(
                size=gamma.shape
            )
        )

        z_prop = (
            z_cur
            +
            U
            @ (
                gamma_prop
                -
                gamma
            )
        )

        yhat_prop = mlp_forward_numpy(
            z_prop,
            fixed_mlp_weights
        )

        ll_prop = gaussian_loglik(
            y,
            yhat_prop,
            sigma2
        )

        gamma_prior_prop = (
            log_gamma_prior(
                gamma_prop
            )
        )

        log_ratio = (
            ll_prop
            +
            gamma_prior_prop
            -
            ll_cur
            -
            gamma_prior_cur
        )

        gamma_accepted = (
            np.log(
                rng.uniform()
            )
            <
            log_ratio
        )

        if it < adapt_stop:

            gamma_window_trials += 1

        if it >= MCMC_BURN:

            gamma_post_trials += 1

        if gamma_accepted:

            gamma = gamma_prop

            z_cur = z_prop

            yhat_cur = yhat_prop

            ll_cur = ll_prop

            gamma_prior_cur = (
                gamma_prior_prop
            )

            if it < adapt_stop:

                gamma_window_accept += 1

            if it >= MCMC_BURN:

                gamma_post_accept += 1


        # ====================================================
        # C. mu block
        # ====================================================

        mu_prop = (
            mu
            +
            mu_step
            * rng.normal(
                size=mu.shape
            )
        )

        z_prop = (
            z_cur
            +
            (
                mu_prop
                -
                mu
            )
        )

        yhat_prop = mlp_forward_numpy(
            z_prop,
            fixed_mlp_weights
        )

        ll_prop = gaussian_loglik(
            y,
            yhat_prop,
            sigma2
        )

        mu_prior_prop = (
            log_mu_prior(
                mu_prop
            )
        )

        log_ratio = (
            ll_prop
            +
            mu_prior_prop
            -
            ll_cur
            -
            mu_prior_cur
        )

        mu_accepted = (
            np.log(
                rng.uniform()
            )
            <
            log_ratio
        )

        if it < adapt_stop:

            mu_window_trials += 1

        if it >= MCMC_BURN:

            mu_post_trials += 1

        if mu_accepted:

            mu = mu_prop

            z_cur = z_prop

            yhat_cur = yhat_prop

            ll_cur = ll_prop

            mu_prior_cur = (
                mu_prior_prop
            )

            if it < adapt_stop:

                mu_window_accept += 1

            if it >= MCMC_BURN:

                mu_post_accept += 1


        # ====================================================
        # D. Gibbs update for sigma^2
        # ====================================================

        residual = (
            y
            -
            yhat_cur
        )

        shape_post = (
            SIGMA_A0
            +
            0.5
            * y.size
        )

        rate_post = (
            SIGMA_B0
            +
            0.5
            * np.sum(
                residual ** 2
            )
        )

        sigma2 = (
            1.0
            /
            rng.gamma(
                shape=shape_post,
                scale=1.0
                /
                rate_post
            )
        )

        ll_cur = gaussian_loglik(
            y,
            yhat_cur,
            sigma2
        )


        # ====================================================
        # E. Proposal adaptation
        #
        # Performed only in the first 80% of burn-in.
        #
        # Separate targets are used for:
        #   Theta : 0.25
        #   Gamma : 0.25
        #   mu    : 0.30
        #
        # The adaptation gain decreases gradually with the
        # adaptation round but remains responsive enough for
        # the short pilot chains.
        # ====================================================

        if (
            it < adapt_stop
            and
            (it + 1)
            %
            MCMC_ADAPT_EVERY
            ==
            0
        ):

            adaptation_round += 1

            adaptive_gain = (
                MCMC_ADAPT_GAIN
                /
                np.sqrt(
                    adaptation_round
                )
            )


            # ------------------------------------------------
            # Theta blocks
            # ------------------------------------------------

            for j in range(M):

                if (
                    theta_window_trials[j]
                    >
                    0
                ):

                    rate = (
                        theta_window_accept[j]
                        /
                        theta_window_trials[j]
                    )

                    theta_steps[j] *= np.exp(
                        adaptive_gain
                        *
                        (
                            rate
                            -
                            MCMC_TARGET_THETA
                        )
                    )

                    theta_steps[j] = float(
                        np.clip(
                            theta_steps[j],
                            THETA_STEP_MIN,
                            THETA_STEP_MAX
                        )
                    )


            # ------------------------------------------------
            # Gamma block
            # ------------------------------------------------

            if gamma_window_trials > 0:

                gamma_rate = (
                    gamma_window_accept
                    /
                    gamma_window_trials
                )

                gamma_step *= np.exp(
                    adaptive_gain
                    *
                    (
                        gamma_rate
                        -
                        MCMC_TARGET_GAMMA
                    )
                )

                gamma_step = float(
                    np.clip(
                        gamma_step,
                        GAMMA_STEP_MIN,
                        GAMMA_STEP_MAX
                    )
                )


            # ------------------------------------------------
            # mu block
            # ------------------------------------------------

            if mu_window_trials > 0:

                mu_rate = (
                    mu_window_accept
                    /
                    mu_window_trials
                )

                mu_step *= np.exp(
                    adaptive_gain
                    *
                    (
                        mu_rate
                        -
                        MCMC_TARGET_MU
                    )
                )

                mu_step = float(
                    np.clip(
                        mu_step,
                        MU_STEP_MIN,
                        MU_STEP_MAX
                    )
                )


            # ------------------------------------------------
            # Reset adaptation windows
            # ------------------------------------------------

            theta_window_accept[:] = 0

            theta_window_trials[:] = 0

            gamma_window_accept = 0

            gamma_window_trials = 0

            mu_window_accept = 0

            mu_window_trials = 0


        # ====================================================
        # F. Save posterior draws
        # ====================================================

        if (
            it
            >=
            MCMC_BURN
            and
            (
                it
                -
                MCMC_BURN
            )
            %
            MCMC_THIN
            ==
            0
        ):

            for j in range(M):

                s = np.linalg.svd(
                    theta[j],
                    compute_uv=False
                )[
                    :K_LIST[j]
                ]

                singular_draws[j].append(
                    s
                )

            sigma_draws.append(
                sigma2
            )


        if (
            heartbeat is not None
            and
            (it + 1) % 50 == 0
        ):

            heartbeat()


    # ========================================================
    # Posterior summaries
    # ========================================================

    singular_draws = [
        np.asarray(
            x
        )
        for x in singular_draws
    ]

    if any(
        len(
            x
        )
        ==
        0
        for x in singular_draws
    ):

        raise RuntimeError(
            "No posterior singular-value draws were retained."
        )

    mean_s2 = [
        np.mean(
            x ** 2,
            axis=0
        )
        for x in singular_draws
    ]


    # ========================================================
    # Post-burn-in acceptance diagnostics
    # ========================================================

    theta_post_acceptance = (
        theta_post_accept
        /
        np.maximum(
            theta_post_trials,
            1
        )
    )

    gamma_post_acceptance = float(
        gamma_post_accept
        /
        max(
            gamma_post_trials,
            1
        )
    )

    mu_post_acceptance = float(
        mu_post_accept
        /
        max(
            mu_post_trials,
            1
        )
    )

    gamma_mu_mean_acceptance = float(
        0.5
        *
        (
            gamma_post_acceptance
            +
            mu_post_acceptance
        )
    )


    return {

        "singular_draws":
            singular_draws,

        "mean_s2":
            mean_s2,

        "sigma_draws":
            np.asarray(
                sigma_draws
            ),

        "theta_acceptance":
            theta_post_acceptance,

        "gamma_acceptance":
            gamma_post_acceptance,

        "mu_acceptance":
            mu_post_acceptance,

        "gm_acceptance":
            gamma_mu_mean_acceptance,

        "theta_steps":
            theta_steps,

        "gamma_step":
            gamma_step,

        "mu_step":
            mu_step
    }


# ============================================================
# 17. STRUCTURAL LOSS
# ============================================================

def tail_loss_arrays(
    mean_s2
):

    losses = []

    for s2 in mean_s2:

        k = len(
            s2
        )

        loss_j = np.array(
            [
                np.sum(
                    s2[
                        d:
                    ]
                )
                for d in range(
                    1,
                    k + 1
                )
            ],
            dtype=float
        )

        losses.append(
            loss_j
        )

    return losses


def allocation_loss(
    d,
    loss_arrays
):

    return float(
        sum(
            loss_arrays[j][
                d[j] - 1
            ]
            for j in range(
                len(
                    d
                )
            )
        )
    )


def embedding_cost(
    d
):

    return int(
        sum(
            N_LEVELS[j]
            *
            d[j]
            for j in range(M)
        )
    )


# ============================================================
# 18. MULTIPLE-CHOICE KNAPSACK DYNAMIC PROGRAMMING
# ============================================================

def dp_allocations(
    n_levels,
    k_list,
    loss_arrays,
    budgets
):

    m = len(
        n_levels
    )

    bmax = max(
        budgets
    )

    dp = np.full(
        (
            m + 1,
            bmax + 1
        ),
        np.inf
    )

    parent_d = np.full(
        (
            m + 1,
            bmax + 1
        ),
        -1,
        dtype=int
    )

    parent_b = np.full(
        (
            m + 1,
            bmax + 1
        ),
        -1,
        dtype=int
    )

    dp[
        0,
        0
    ] = 0.0


    for j in range(
        1,
        m + 1
    ):

        Nj = n_levels[
            j - 1
        ]

        Kj = k_list[
            j - 1
        ]

        for b in range(
            bmax + 1
        ):

            for d in range(
                1,
                Kj + 1
            ):

                cost = (
                    Nj
                    *
                    d
                )

                if (
                    b
                    >=
                    cost
                    and
                    np.isfinite(
                        dp[
                            j - 1,
                            b - cost
                        ]
                    )
                ):

                    value = (
                        dp[
                            j - 1,
                            b - cost
                        ]
                        +
                        loss_arrays[
                            j - 1
                        ][
                            d - 1
                        ]
                    )

                    if (
                        value
                        <
                        dp[
                            j,
                            b
                        ]
                    ):

                        dp[
                            j,
                            b
                        ] = value

                        parent_d[
                            j,
                            b
                        ] = d

                        parent_b[
                            j,
                            b
                        ] = (
                            b
                            -
                            cost
                        )


    out = {}


    for B in budgets:

        feasible_costs = np.arange(
            B + 1
        )

        values = dp[
            m,
            feasible_costs
        ]

        finite_mask = np.isfinite(
            values
        )

        if not np.any(
            finite_mask
        ):

            raise RuntimeError(
                f"No feasible allocation for budget {B}."
            )

        feasible_indices = feasible_costs[
            finite_mask
        ]

        finite_values = values[
            finite_mask
        ]

        best_cost = int(
            feasible_indices[
                np.argmin(
                    finite_values
                )
            ]
        )

        dvec = [
            None
        ] * m

        b = best_cost

        for j in range(
            m,
            0,
            -1
        ):

            d = int(
                parent_d[
                    j,
                    b
                ]
            )

            if d < 1:

                raise RuntimeError(
                    "DP backtracking failed."
                )

            dvec[
                j - 1
            ] = d

            b = int(
                parent_b[
                    j,
                    b
                ]
            )

        out[
            B
        ] = {

            "d":
                tuple(
                    dvec
                ),

            "loss":
                float(
                    dp[
                        m,
                        best_cost
                    ]
                ),

            "actual_cost":
                best_cost
        }

    return out


# ============================================================
# 19. REDUCED CANDIDATE MODEL
# ============================================================

class ReducedModel(
    nn.Module
):

    def __init__(
        self,
        d
    ):

        super().__init__()

        self.d = list(
            d
        )

        self.embeddings = nn.ModuleList(
            [
                nn.Embedding(
                    N_LEVELS[j],
                    self.d[j]
                )
                for j in range(M)
            ]
        )

        for emb in self.embeddings:

            nn.init.normal_(
                emb.weight,
                std=0.10
            )

        self.gamma = nn.Parameter(
            0.10
            *
            torch.randn(
                P_U,
                P_Z
            )
        )

        self.mu = nn.Parameter(
            torch.zeros(
                P_Z
            )
        )

        input_dim = (
            sum(
                self.d
            )
            +
            P_Z
        )

        self.mlp = nn.Sequential(

            nn.Linear(
                input_dim,
                RED_HIDDEN[
                    0
                ]
            ),

            nn.Tanh(),

            nn.Linear(
                RED_HIDDEN[
                    0
                ],
                RED_HIDDEN[
                    1
                ]
            ),

            nn.Tanh(),

            nn.Linear(
                RED_HIDDEN[
                    1
                ],
                Q
            )
        )

    def latent(
        self,
        C,
        U
    ):

        pieces = [
            self.embeddings[j](
                C[j]
            )
            for j in range(M)
        ]

        continuous = (
            self.mu.unsqueeze(
                0
            )
            +
            U
            @ self.gamma
        )

        pieces.append(
            continuous
        )

        return torch.cat(
            pieces,
            dim=1
        )

    def forward(
        self,
        C,
        U
    ):

        return self.mlp(
            self.latent(
                C,
                U
            )
        )

    @torch.no_grad()
    def center_embeddings_(
        self
    ):

        for emb in self.embeddings:

            emb.weight -= (
                emb.weight.mean(
                    dim=0,
                    keepdim=True
                )
            )


# ============================================================
# 20. REDUCED-MODEL TRAINING
# ============================================================

def train_reduced_once(
    C,
    U,
    y,
    d,
    seed,
    heartbeat=None
):

    set_seed(
        seed
    )

    model = ReducedModel(
        d
    ).to(
        DEVICE
    )

    C_t = [
        torch.tensor(
            x,
            dtype=torch.long,
            device=DEVICE
        )
        for x in C
    ]

    U_t = torch.tensor(
        U,
        dtype=torch.float32,
        device=DEVICE
    )

    y_t = torch.tensor(
        y,
        dtype=torch.float32,
        device=DEVICE
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.005,
        weight_decay=RED_WEIGHT_DECAY
    )


    for epoch in range(
        RED_EPOCHS
    ):

        optimizer.zero_grad()

        pred = model(
            C_t,
            U_t
        )

        loss = F.mse_loss(
            pred,
            y_t
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0
        )

        optimizer.step()

        model.center_embeddings_()

        if (
            heartbeat is not None
            and
            (epoch + 1) % 50 == 0
        ):

            heartbeat()


    with torch.no_grad():

        final_loss = float(
            F.mse_loss(
                model(
                    C_t,
                    U_t
                ),
                y_t
            ).cpu()
        )

    return (
        model,
        final_loss
    )


def train_reduced_model(
    C,
    U,
    y,
    d,
    seed,
    heartbeat=None
):

    best_model = None

    best_loss = np.inf


    for r in range(
        RED_RESTARTS
    ):

        model, loss = (
            train_reduced_once(
                C,
                U,
                y,
                d,
                seed
                +
                1000
                * r,
                heartbeat
            )
        )

        if loss < best_loss:

            if best_model is not None:

                del best_model

            best_model = model

            best_loss = loss

        else:

            del model

        gc.collect()


    return (
        best_model,
        best_loss
    )


@torch.no_grad()
def predict_reduced(
    model,
    C,
    U
):

    C_t = [
        torch.tensor(
            x,
            dtype=torch.long,
            device=DEVICE
        )
        for x in C
    ]

    U_t = torch.tensor(
        U,
        dtype=torch.float32,
        device=DEVICE
    )

    return (
        model(
            C_t,
            U_t
        )
        .cpu()
        .numpy()
    )


def mse(
    y,
    pred
):

    return float(
        np.mean(
            (
                y
                -
                pred
            ) ** 2
        )
    )


# ============================================================
# 21. POSTERIOR SPECTRAL DIAGNOSTICS
# ============================================================

def posterior_spectral_diagnostics(
    singular_draws,
    true_singular_values
):

    s2_errors = []

    tail_errors = []

    tail_widths = []


    for j in range(M):

        draws = singular_draws[
            j
        ]

        posterior_mean_s2 = np.mean(
            draws ** 2,
            axis=0
        )

        true_s2 = (
            np.asarray(
                true_singular_values[
                    j
                ]
            ) ** 2
        )

        s2_errors.extend(
            (
                posterior_mean_s2
                -
                true_s2
            ).tolist()
        )


        Kj = len(
            true_s2
        )

        for d in range(
            1,
            Kj
        ):

            posterior_tail = np.sum(
                draws[
                    :,
                    d:
                ] ** 2,
                axis=1
            )

            posterior_tail_mean = float(
                np.mean(
                    posterior_tail
                )
            )

            true_tail = float(
                np.sum(
                    true_s2[
                        d:
                    ]
                )
            )

            tail_errors.append(
                posterior_tail_mean
                -
                true_tail
            )

            lower, upper = np.quantile(
                posterior_tail,
                [
                    0.025,
                    0.975
                ]
            )

            tail_widths.append(
                upper
                -
                lower
            )


    s2_rmse = float(
        np.sqrt(
            np.mean(
                np.asarray(
                    s2_errors
                ) ** 2
            )
        )
    )

    tail_rmse = float(
        np.sqrt(
            np.mean(
                np.asarray(
                    tail_errors
                ) ** 2
            )
        )
    )

    mean_tail_width = float(
        np.mean(
            tail_widths
        )
    )


    return (
        s2_rmse,
        tail_rmse,
        mean_tail_width
    )


# ============================================================
# 22. ENUMERATION FOR STRUCTURAL SCORE ERROR
# ============================================================

def all_allocations():

    return itertools.product(
        *[
            range(
                1,
                K_LIST[j] + 1
            )
            for j in range(M)
        ]
    )


def compute_delta_max(
    posterior_losses,
    true_losses
):

    delta = 0.0


    for d in all_allocations():

        posterior_value = allocation_loss(
            d,
            posterior_losses
        )

        true_value = allocation_loss(
            d,
            true_losses
        )

        delta = max(
            delta,
            abs(
                posterior_value
                -
                true_value
            )
        )


    return float(
        delta
    )


def compute_normalized_delta(
    posterior_losses,
    true_losses
):

    delta = compute_delta_max(
        posterior_losses,
        true_losses
    )

    baseline_d = tuple(
        1
        for _ in range(M)
    )

    scale = allocation_loss(
        baseline_d,
        true_losses
    )

    return float(
        delta
        /
        max(
            scale,
            1e-12
        )
    )


# ============================================================
# 23. CHECKPOINT
# ============================================================

if (
    MODE == "2"
    and
    os.path.exists(
        CHECKPOINT_PATH
    )
):

    with open(
        CHECKPOINT_PATH,
        "rb"
    ) as f:

        STATE = pickle.load(
            f
        )

    if (
        STATE.get(
            "config_signature"
        )
        !=
        CONFIG_SIGNATURE
    ):

        raise RuntimeError(
            "The checkpoint was created using a different "
            "configuration. Start from the beginning or restore "
            "the previous configuration."
        )

    print(
        "Resumed checkpoint with",
        len(
            STATE[
                "completed"
            ]
        ),
        "completed replications."
    )

else:

    STATE = {

        "config_signature":
            CONFIG_SIGNATURE,

        "completed":
            set(),

        "replication_rows":
            [],

        "budget_rows":
            [],

        "started":
            time.time()
    }

    if MODE == "2":

        print(
            "No compatible checkpoint was found. "
            "Starting from the beginning."
        )


CHECKPOINT_CLOCK = {
    "last":
        time.time()
}


def save_tables():

    if (
        len(
            STATE[
                "replication_rows"
            ]
        )
        >
        0
    ):

        pd.DataFrame(
            STATE[
                "replication_rows"
            ]
        ).to_csv(
            RESULTS_PATH,
            index=False
        )

    if (
        len(
            STATE[
                "budget_rows"
            ]
        )
        >
        0
    ):

        pd.DataFrame(
            STATE[
                "budget_rows"
            ]
        ).to_csv(
            BUDGET_RESULTS_PATH,
            index=False
        )


def save_checkpoint(
    force=False
):

    now = time.time()

    if (
        force
        or
        now
        -
        CHECKPOINT_CLOCK[
            "last"
        ]
        >=
        CHECKPOINT_SECONDS
    ):

        temporary_path = (
            CHECKPOINT_PATH
            +
            ".tmp"
        )

        with open(
            temporary_path,
            "wb"
        ) as f:

            pickle.dump(
                STATE,
                f,
                protocol=pickle.HIGHEST_PROTOCOL
            )

        os.replace(
            temporary_path,
            CHECKPOINT_PATH
        )

        save_tables()

        CHECKPOINT_CLOCK[
            "last"
        ] = now

        print(
            "[checkpoint] saved at",
            time.strftime(
                "%H:%M:%S"
            )
        )


# ============================================================
# 24. SINGLE REPLICATION
# ============================================================

def run_replication(
    scenario,
    scenario_index,
    rep
):

    t0 = time.time()

    seed = (
        BASE_SEED
        +
        100000
        *
        scenario_index
        +
        rep
    )


    # --------------------------------------------------------
    # Truth
    # --------------------------------------------------------

    true_par = build_true_parameters(
        scenario[
            "spectral_regime"
        ]
    )


    # --------------------------------------------------------
    # Independent train / validation / test sets
    # --------------------------------------------------------

    Ctr, Utr, ytr, _ = generate_dataset(
        scenario[
            "n_train"
        ],
        true_par,
        scenario[
            "noise_sd"
        ],
        seed + 11
    )

    Cva, Uva, yva, _ = generate_dataset(
        N_VAL,
        true_par,
        scenario[
            "noise_sd"
        ],
        seed + 22
    )

    Cte, Ute, yte, _ = generate_dataset(
        N_TEST,
        true_par,
        scenario[
            "noise_sd"
        ],
        seed + 33
    )


    # --------------------------------------------------------
    # 1. Rich reference model
    # --------------------------------------------------------

    reference, reference_train_loss = (
        train_reference_model(
            Ctr,
            Utr,
            ytr,
            seed + 100,
            heartbeat=lambda:
                save_checkpoint(
                    force=False
                )
        )
    )


    reference_train_pred = predict_reference(
        reference,
        Ctr,
        Utr
    )

    reference_test_pred = predict_reference(
        reference,
        Cte,
        Ute
    )

    reference_test_mse = mse(
        yte,
        reference_test_pred
    )


    (
        theta_init,
        gamma_init,
        mu_init,
        omega_fixed
    ) = extract_reference_parameters(
        reference
    )


    sigma2_init = max(
        mse(
            ytr,
            reference_train_pred
        ),
        1e-6
    )


    # --------------------------------------------------------
    # 2. Bayesian fixed-dimensional inference
    # --------------------------------------------------------

    posterior = blocked_mcmc(

        Ctr,

        Utr,

        ytr,

        theta_init,

        gamma_init,

        mu_init,

        omega_fixed,

        sigma2_init,

        seed + 200,

        heartbeat=lambda:
            save_checkpoint(
                force=False
            )
    )


    posterior_losses = tail_loss_arrays(
        posterior[
            "mean_s2"
        ]
    )


    # --------------------------------------------------------
    # 3. True population structural criterion
    # --------------------------------------------------------

    true_s2 = [
        np.asarray(
            s
        ) ** 2
        for s in true_par[
            "singular_values"
        ]
    ]

    true_losses = tail_loss_arrays(
        true_s2
    )


    # --------------------------------------------------------
    # Posterior and oracle DP allocations
    # --------------------------------------------------------

    posterior_dp = dp_allocations(
        N_LEVELS,
        K_LIST,
        posterior_losses,
        BUDGETS
    )

    oracle_dp = dp_allocations(
        N_LEVELS,
        K_LIST,
        true_losses,
        BUDGETS
    )


    # --------------------------------------------------------
    # Spectral diagnostics
    # --------------------------------------------------------

    (
        posterior_s2_rmse,
        posterior_tail_rmse,
        posterior_tail_interval_width
    ) = posterior_spectral_diagnostics(
        posterior[
            "singular_draws"
        ],
        true_par[
            "singular_values"
        ]
    )


    delta_max = compute_delta_max(
        posterior_losses,
        true_losses
    )

    normalized_delta = compute_normalized_delta(
        posterior_losses,
        true_losses
    )


    # --------------------------------------------------------
    # Refit ONLY structurally selected allocations
    # --------------------------------------------------------

    unique_allocations = sorted(
        set(
            posterior_dp[
                B
            ][
                "d"
            ]
            for B in BUDGETS
        )
    )


    candidate_cache = {}


    for candidate_index, d in enumerate(
        unique_allocations
    ):

        candidate_seed = (
            seed
            +
            10000
            +
            100
            *
            candidate_index
            +
            sum(
                (
                    j + 1
                )
                *
                d[j]
                for j in range(M)
            )
        )

        reduced, training_loss = (
            train_reduced_model(

                Ctr,

                Utr,

                ytr,

                d,

                candidate_seed,

                heartbeat=lambda:
                    save_checkpoint(
                        force=False
                    )
            )
        )


        validation_pred = predict_reduced(
            reduced,
            Cva,
            Uva
        )

        test_pred = predict_reduced(
            reduced,
            Cte,
            Ute
        )


        candidate_cache[
            d
        ] = {

            "train_mse":
                training_loss,

            "val_mse":
                mse(
                    yva,
                    validation_pred
                ),

            "test_mse":
                mse(
                    yte,
                    test_pred
                ),

            "total_parameters":
                int(
                    sum(
                        p.numel()
                        for p in reduced.parameters()
                    )
                )
        }


        del reduced

        gc.collect()


    # --------------------------------------------------------
    # 4. Budget-level structural diagnostics
    # --------------------------------------------------------

    budget_rows = []

    exact_recovery_indicators = []

    allocation_l1_errors = []

    structural_excesses = []


    for B in BUDGETS:

        d_post = posterior_dp[
            B
        ][
            "d"
        ]

        d_oracle = oracle_dp[
            B
        ][
            "d"
        ]

        candidate = candidate_cache[
            d_post
        ]


        true_structural_loss_selected = allocation_loss(
            d_post,
            true_losses
        )

        true_structural_loss_oracle = allocation_loss(
            d_oracle,
            true_losses
        )


        structural_excess = (
            true_structural_loss_selected
            -
            true_structural_loss_oracle
        )


        exact_recovery = float(
            tuple(
                d_post
            )
            ==
            tuple(
                d_oracle
            )
        )


        l1_error = float(
            np.mean(
                np.abs(
                    np.asarray(
                        d_post
                    )
                    -
                    np.asarray(
                        d_oracle
                    )
                )
            )
        )


        exact_recovery_indicators.append(
            exact_recovery
        )

        allocation_l1_errors.append(
            l1_error
        )

        structural_excesses.append(
            structural_excess
        )


        budget_rows.append(
            {

                "scenario":
                    scenario[
                        "name"
                    ],

                "spectral_regime":
                    scenario[
                        "spectral_regime"
                    ],

                "n_train":
                    scenario[
                        "n_train"
                    ],

                "rep":
                    rep,

                "B":
                    B,

                "posterior_d":
                    str(
                        tuple(
                            d_post
                        )
                    ),

                "oracle_d":
                    str(
                        tuple(
                            d_oracle
                        )
                    ),

                "actual_cost":
                    embedding_cost(
                        d_post
                    ),

                "posterior_structural_score":
                    posterior_dp[
                        B
                    ][
                        "loss"
                    ],

                "true_structural_loss":
                    true_structural_loss_selected,

                "oracle_structural_loss":
                    true_structural_loss_oracle,

                "structural_excess":
                    structural_excess,

                "exact_allocation_recovery":
                    exact_recovery,

                "allocation_l1_error":
                    l1_error,

                "training_mse":
                    candidate[
                        "train_mse"
                    ],

                "validation_mse":
                    candidate[
                        "val_mse"
                    ],

                "test_mse":
                    candidate[
                        "test_mse"
                    ]
            }
        )


    # --------------------------------------------------------
    # 5. Adaptive budget selection
    # --------------------------------------------------------

    BMAX = max(
        BUDGETS
    )

    d_bmax = posterior_dp[
        BMAX
    ][
        "d"
    ]

    max_budget_val = candidate_cache[
        d_bmax
    ][
        "val_mse"
    ]


    admissible_budgets = [
        B
        for B in sorted(
            BUDGETS
        )
        if (
            candidate_cache[
                posterior_dp[
                    B
                ][
                    "d"
                ]
            ][
                "val_mse"
            ]
            <=
            max_budget_val
            +
            ZETA_N
        )
    ]


    B_hat = min(
        admissible_budgets
    )

    d_hat = posterior_dp[
        B_hat
    ][
        "d"
    ]


    final_candidate = candidate_cache[
        d_hat
    ]

    max_candidate = candidate_cache[
        d_bmax
    ]


    # --------------------------------------------------------
    # Large-test-sample oracle budget
    # --------------------------------------------------------

    max_budget_test = candidate_cache[
        d_bmax
    ][
        "test_mse"
    ]


    test_oracle_admissible = [
        B
        for B in sorted(
            BUDGETS
        )
        if (
            candidate_cache[
                posterior_dp[
                    B
                ][
                    "d"
                ]
            ][
                "test_mse"
            ]
            <=
            max_budget_test
            +
            ZETA_N
        )
    ]


    B_test_oracle = min(
        test_oracle_admissible
    )


    budget_exact_match = float(
        B_hat
        ==
        B_test_oracle
    )


    budget_absolute_error = float(
        abs(
            B_hat
            -
            B_test_oracle
        )
    )


    # --------------------------------------------------------
    # Parameter reduction
    # --------------------------------------------------------

    rich_embedding_parameters = sum(
        N_LEVELS[j]
        *
        P_Z
        for j in range(M)
    )


    selected_embedding_parameters = embedding_cost(
        d_hat
    )


    embedding_reduction = (
        1.0
        -
        selected_embedding_parameters
        /
        rich_embedding_parameters
    )


    # --------------------------------------------------------
    # Runtime
    # --------------------------------------------------------

    runtime_seconds = (
        time.time()
        -
        t0
    )


    # --------------------------------------------------------
    # Replication-level result
    # --------------------------------------------------------

    rep_row = {

        "scenario":
            scenario[
                "name"
            ],

        "spectral_regime":
            scenario[
                "spectral_regime"
            ],

        "n_train":
            scenario[
                "n_train"
            ],

        "rep":
            rep,

        "noise_sd":
            scenario[
                "noise_sd"
            ],

        "selected_budget":
            B_hat,

        "test_oracle_budget":
            B_test_oracle,

        "budget_exact_match":
            budget_exact_match,

        "budget_absolute_error":
            budget_absolute_error,

        "selected_d":
            str(
                tuple(
                    d_hat
                )
            ),

        "selected_embedding_parameters":
            selected_embedding_parameters,

        "rich_embedding_parameters":
            rich_embedding_parameters,

        "embedding_reduction":
            embedding_reduction,

        "allocation_recovery_rate":
            float(
                np.mean(
                    exact_recovery_indicators
                )
            ),

        "allocation_mean_l1_error":
            float(
                np.mean(
                    allocation_l1_errors
                )
            ),

        "mean_structural_excess":
            float(
                np.mean(
                    structural_excesses
                )
            ),

        "delta_score_error":
            delta_max,

        "normalized_delta_score_error":
            normalized_delta,

        "posterior_s2_rmse":
            posterior_s2_rmse,

        "posterior_tail_rmse":
            posterior_tail_rmse,

        "posterior_tail_interval_width":
            posterior_tail_interval_width,

        "reference_train_mse":
            reference_train_loss,

        "reference_test_mse":
            reference_test_mse,

        "final_validation_mse":
            final_candidate[
                "val_mse"
            ],

        "final_test_mse":
            final_candidate[
                "test_mse"
            ],

        "max_budget_test_mse":
            max_candidate[
                "test_mse"
            ],

        "test_excess_vs_max_budget":
            (
                final_candidate[
                    "test_mse"
                ]
                -
                max_candidate[
                    "test_mse"
                ]
            ),

        # ----------------------------------------------------
        # Post-burn-in MCMC diagnostics
        # ----------------------------------------------------

        "mean_theta_acceptance":
            float(
                np.mean(
                    posterior[
                        "theta_acceptance"
                    ]
                )
            ),

        "min_theta_acceptance":
            float(
                np.min(
                    posterior[
                        "theta_acceptance"
                    ]
                )
            ),

        "max_theta_acceptance":
            float(
                np.max(
                    posterior[
                        "theta_acceptance"
                    ]
                )
            ),

        "gamma_acceptance":
            float(
                posterior[
                    "gamma_acceptance"
                ]
            ),

        "mu_acceptance":
            float(
                posterior[
                    "mu_acceptance"
                ]
            ),

        "gamma_mu_acceptance":
            float(
                posterior[
                    "gm_acceptance"
                ]
            ),

        "final_gamma_step":
            float(
                posterior[
                    "gamma_step"
                ]
            ),

        "final_mu_step":
            float(
                posterior[
                    "mu_step"
                ]
            ),

        "n_unique_refits":
            len(
                unique_allocations
            ),

        "runtime_seconds":
            runtime_seconds
    }


    del reference

    gc.collect()


    return (
        rep_row,
        budget_rows
    )


# ============================================================
# 25. RUN SIMULATION
# ============================================================

print(
    "\n"
    "SIMULATION 1 — MCMC-TUNED PILOT\n"
    "---------------------------------------------"
)

print(
    "Spectral regimes:",
    [
        "clear",
        "weak"
    ]
)

print(
    "Training sizes:",
    [
        250,
        500,
        1000
    ]
)

print(
    "N_j:",
    N_LEVELS
)

print(
    "K_j:",
    K_LIST
)

print(
    "Candidate budgets:",
    BUDGETS
)

print(
    "Minimum budget:",
    MIN_COST
)

print(
    "Maximum budget:",
    MAX_COST
)

print(
    "Budget tolerance zeta_n:",
    ZETA_N
)

print(
    "Noise SD:",
    NOISE_SD
)

print(
    "Monte Carlo replications per scenario:",
    N_REPS
)

print(
    "Pilot mode:",
    PILOT_MODE
)

print(
    "MCMC targets: theta =",
    MCMC_TARGET_THETA,
    ", gamma =",
    MCMC_TARGET_GAMMA,
    ", mu =",
    MCMC_TARGET_MU
)

print(
    "---------------------------------------------\n"
)


try:

    for scenario_index, scenario in enumerate(
        SCENARIOS
    ):

        for rep in range(
            1,
            N_REPS + 1
        ):

            task_key = (
                scenario[
                    "name"
                ],
                rep
            )


            if (
                task_key
                in
                STATE[
                    "completed"
                ]
            ):

                continue


            print(
                f"\n[{scenario['name']}] "
                f"replication {rep}/{N_REPS}"
            )


            rep_row, budget_rows = run_replication(
                scenario,
                scenario_index,
                rep
            )


            STATE[
                "replication_rows"
            ].append(
                rep_row
            )


            STATE[
                "budget_rows"
            ].extend(
                budget_rows
            )


            STATE[
                "completed"
            ].add(
                task_key
            )


            save_checkpoint(
                force=True
            )


            print(
                "  B_hat =",
                rep_row[
                    "selected_budget"
                ],
                "| test-oracle B =",
                rep_row[
                    "test_oracle_budget"
                ],
                "| d =",
                rep_row[
                    "selected_d"
                ],
                "| recovery =",
                round(
                    rep_row[
                        "allocation_recovery_rate"
                    ],
                    3
                ),
                "| delta_norm =",
                round(
                    rep_row[
                        "normalized_delta_score_error"
                    ],
                    3
                ),
                "| test MSE =",
                round(
                    rep_row[
                        "final_test_mse"
                    ],
                    5
                )
            )

            print(
                "  post-burn acceptance:"
                " theta =",
                round(
                    rep_row[
                        "mean_theta_acceptance"
                    ],
                    3
                ),
                "| gamma =",
                round(
                    rep_row[
                        "gamma_acceptance"
                    ],
                    3
                ),
                "| mu =",
                round(
                    rep_row[
                        "mu_acceptance"
                    ],
                    3
                )
            )


except KeyboardInterrupt:

    print(
        "\nInterrupted by user. "
        "Saving checkpoint..."
    )

    save_checkpoint(
        force=True
    )

    raise


except Exception:

    print(
        "\nAn error occurred. "
        "Saving checkpoint..."
    )

    save_checkpoint(
        force=True
    )

    raise


# ============================================================
# 26. SAVE FINAL RESULTS
# ============================================================

rep_df = pd.DataFrame(
    STATE[
        "replication_rows"
    ]
)

budget_df = pd.DataFrame(
    STATE[
        "budget_rows"
    ]
)


rep_df.to_csv(
    RESULTS_PATH,
    index=False
)

budget_df.to_csv(
    BUDGET_RESULTS_PATH,
    index=False
)


# ============================================================
# 27. MAIN SUMMARY
# ============================================================

summary = (
    rep_df
    .groupby(
        [
            "spectral_regime",
            "n_train"
        ]
    )
    .agg(

        n_rep=(
            "rep",
            "count"
        ),

        allocation_recovery=(
            "allocation_recovery_rate",
            "mean"
        ),

        allocation_l1_error=(
            "allocation_mean_l1_error",
            "mean"
        ),

        structural_excess=(
            "mean_structural_excess",
            "mean"
        ),

        delta_score_error=(
            "delta_score_error",
            "mean"
        ),

        normalized_delta=(
            "normalized_delta_score_error",
            "mean"
        ),

        posterior_s2_rmse=(
            "posterior_s2_rmse",
            "mean"
        ),

        posterior_tail_rmse=(
            "posterior_tail_rmse",
            "mean"
        ),

        posterior_tail_width=(
            "posterior_tail_interval_width",
            "mean"
        ),

        mean_selected_budget=(
            "selected_budget",
            "mean"
        ),

        mean_test_oracle_budget=(
            "test_oracle_budget",
            "mean"
        ),

        budget_match_rate=(
            "budget_exact_match",
            "mean"
        ),

        budget_absolute_error=(
            "budget_absolute_error",
            "mean"
        ),

        embedding_reduction=(
            "embedding_reduction",
            "mean"
        ),

        final_test_mse=(
            "final_test_mse",
            "mean"
        ),

        max_budget_test_mse=(
            "max_budget_test_mse",
            "mean"
        ),

        reference_test_mse=(
            "reference_test_mse",
            "mean"
        ),

        test_excess_vs_max_budget=(
            "test_excess_vs_max_budget",
            "mean"
        ),

        mean_theta_acceptance=(
            "mean_theta_acceptance",
            "mean"
        ),

        gamma_acceptance=(
            "gamma_acceptance",
            "mean"
        ),

        mu_acceptance=(
            "mu_acceptance",
            "mean"
        ),

        gamma_mu_acceptance=(
            "gamma_mu_acceptance",
            "mean"
        ),

        mean_final_gamma_step=(
            "final_gamma_step",
            "mean"
        ),

        mean_final_mu_step=(
            "final_mu_step",
            "mean"
        ),

        mean_unique_refits=(
            "n_unique_refits",
            "mean"
        ),

        runtime_seconds=(
            "runtime_seconds",
            "mean"
        )
    )
    .reset_index()
)


# ============================================================
# 28. STANDARD ERRORS
# ============================================================

def standard_error(
    x
):

    x = np.asarray(
        x,
        dtype=float
    )

    x = x[
        np.isfinite(
            x
        )
    ]

    if len(
        x
    ) <= 1:

        return np.nan

    return float(
        np.std(
            x,
            ddof=1
        )
        /
        np.sqrt(
            len(
                x
            )
        )
    )


se_rows = []


for (
    spectral_regime,
    n_train
), tmp in rep_df.groupby(
    [
        "spectral_regime",
        "n_train"
    ]
):

    se_rows.append(
        {

            "spectral_regime":
                spectral_regime,

            "n_train":
                n_train,

            "allocation_recovery_se":
                standard_error(
                    tmp[
                        "allocation_recovery_rate"
                    ]
                ),

            "allocation_l1_error_se":
                standard_error(
                    tmp[
                        "allocation_mean_l1_error"
                    ]
                ),

            "normalized_delta_se":
                standard_error(
                    tmp[
                        "normalized_delta_score_error"
                    ]
                ),

            "selected_budget_se":
                standard_error(
                    tmp[
                        "selected_budget"
                    ]
                ),

            "final_test_mse_se":
                standard_error(
                    tmp[
                        "final_test_mse"
                    ]
                ),

            "theta_acceptance_se":
                standard_error(
                    tmp[
                        "mean_theta_acceptance"
                    ]
                ),

            "gamma_acceptance_se":
                standard_error(
                    tmp[
                        "gamma_acceptance"
                    ]
                ),

            "mu_acceptance_se":
                standard_error(
                    tmp[
                        "mu_acceptance"
                    ]
                )
        }
    )


se_df = pd.DataFrame(
    se_rows
)


summary = summary.merge(
    se_df,
    on=[
        "spectral_regime",
        "n_train"
    ],
    how="left"
)


summary.to_csv(
    SUMMARY_PATH,
    index=False
)


# ============================================================
# 29. FIGURES
# ============================================================

# ------------------------------------------------------------
# Figure 1:
# Allocation recovery versus n
# ------------------------------------------------------------

plt.figure(
    figsize=(
        8,
        5
    )
)

for regime in [
    "clear",
    "weak"
]:

    tmp = summary[
        summary[
            "spectral_regime"
        ]
        ==
        regime
    ].sort_values(
        "n_train"
    )

    plt.plot(
        tmp[
            "n_train"
        ],
        tmp[
            "allocation_recovery"
        ],
        marker="o",
        label=regime
    )

plt.xlabel(
    "Training sample size"
)

plt.ylabel(
    "Exact allocation recovery rate"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        ROOT,
        "sim1_recovery_vs_n.png"
    ),
    dpi=180
)

plt.close()


# ------------------------------------------------------------
# Figure 2:
# Normalized structural-score error versus n
# ------------------------------------------------------------

plt.figure(
    figsize=(
        8,
        5
    )
)

for regime in [
    "clear",
    "weak"
]:

    tmp = summary[
        summary[
            "spectral_regime"
        ]
        ==
        regime
    ].sort_values(
        "n_train"
    )

    plt.plot(
        tmp[
            "n_train"
        ],
        tmp[
            "normalized_delta"
        ],
        marker="o",
        label=regime
    )

plt.xlabel(
    "Training sample size"
)

plt.ylabel(
    "Normalized structural-score discrepancy"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        ROOT,
        "sim1_delta_vs_n.png"
    ),
    dpi=180
)

plt.close()


# ------------------------------------------------------------
# Figure 3:
# Selected budget versus n
# ------------------------------------------------------------

plt.figure(
    figsize=(
        8,
        5
    )
)

for regime in [
    "clear",
    "weak"
]:

    tmp = summary[
        summary[
            "spectral_regime"
        ]
        ==
        regime
    ].sort_values(
        "n_train"
    )

    plt.plot(
        tmp[
            "n_train"
        ],
        tmp[
            "mean_selected_budget"
        ],
        marker="o",
        label=regime
    )

plt.xlabel(
    "Training sample size"
)

plt.ylabel(
    "Mean selected budget"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        ROOT,
        "sim1_selected_budget.png"
    ),
    dpi=180
)

plt.close()


# ------------------------------------------------------------
# Figure 4:
# Selected versus large-test oracle budget
# ------------------------------------------------------------

plt.figure(
    figsize=(
        8,
        5
    )
)

x_labels = [
    f"{r}\n$n={n}$"
    for r, n in zip(
        summary[
            "spectral_regime"
        ],
        summary[
            "n_train"
        ]
    )
]

x = np.arange(
    len(
        summary
    )
)

width = 0.35

plt.bar(
    x
    -
    width / 2,
    summary[
        "mean_selected_budget"
    ],
    width,
    label="Validation-selected"
)

plt.bar(
    x
    +
    width / 2,
    summary[
        "mean_test_oracle_budget"
    ],
    width,
    label="Large-test oracle"
)

plt.xticks(
    x,
    x_labels
)

plt.ylabel(
    "Budget"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        ROOT,
        "sim1_budget_oracle_comparison.png"
    ),
    dpi=180
)

plt.close()


# ------------------------------------------------------------
# Figure 5:
# Predictive performance
# ------------------------------------------------------------

plt.figure(
    figsize=(
        9,
        5
    )
)

x = np.arange(
    len(
        summary
    )
)

width = 0.30

plt.bar(
    x
    -
    width,
    summary[
        "final_test_mse"
    ],
    width,
    label="Adaptive"
)

plt.bar(
    x,
    summary[
        "max_budget_test_mse"
    ],
    width,
    label="Maximum budget"
)

plt.bar(
    x
    +
    width,
    summary[
        "reference_test_mse"
    ],
    width,
    label="Reference model"
)

plt.xticks(
    x,
    x_labels
)

plt.ylabel(
    "Test MSE"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        ROOT,
        "sim1_predictive_performance.png"
    ),
    dpi=180
)

plt.close()


# ------------------------------------------------------------
# Figure 6:
# Post-burn-in MCMC acceptance rates
# ------------------------------------------------------------

plt.figure(
    figsize=(
        9,
        5
    )
)

x = np.arange(
    len(
        summary
    )
)

width = 0.25

plt.bar(
    x - width,
    summary[
        "mean_theta_acceptance"
    ],
    width,
    label=r"$\Theta$"
)

plt.bar(
    x,
    summary[
        "gamma_acceptance"
    ],
    width,
    label=r"$\Gamma$"
)

plt.bar(
    x + width,
    summary[
        "mu_acceptance"
    ],
    width,
    label=r"$\mu$"
)

plt.axhline(
    0.15,
    color="grey",
    linestyle="--",
    linewidth=1
)

plt.axhline(
    0.35,
    color="grey",
    linestyle="--",
    linewidth=1
)

plt.xticks(
    x,
    x_labels
)

plt.ylabel(
    "Post-burn-in acceptance rate"
)

plt.ylim(
    0,
    0.6
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        ROOT,
        "sim1_mcmc_acceptance.png"
    ),
    dpi=180
)

plt.close()


# ============================================================
# 30. FINAL CHECKPOINT
# ============================================================

STATE[
    "finished"
] = True

save_checkpoint(
    force=True
)


# ============================================================
# 31. OUTPUT
# ============================================================

print(
    "\nSIMULATION 1 MCMC-TUNED PILOT COMPLETED."
)

print(
    "\nMain summary:\n"
)

print(
    summary.to_string(
        index=False
    )
)

print(
    "\n\nCHECK BEFORE THE FINAL 100-REPLICATION RUN:"
)

print(
    "\n1. Statistical behavior should remain essentially unchanged:"
)

print(
    "   recovery should increase with n and normalized delta should decrease."
)

print(
    "\n2. Clear spectra should continue to select substantially smaller budgets."
)

print(
    "\n3. Weak spectra should continue to retain the full or near-full budget."
)

print(
    "\n4. Adaptive test MSE should remain close to maximum-budget test MSE."
)

print(
    "\n5. POST-BURN-IN acceptance rates should preferably lie roughly in:"
)

print(
    "   0.15--0.35 for Theta, Gamma and mu."
)

print(
    "\n6. In particular, Gamma/mu acceptance at n=1000 should no longer collapse."
)

print(
    "\nIf these conditions hold, set PILOT_MODE=False and run the final experiment."
)

print(
    "\nFiles saved in:"
)

print(
    ROOT
)

Simulation 1 rounded-final: type 1 to START FROM THE BEGINNING or 2 to RESUME FROM THE LAST CHECKPOINT: 2
Mounted at /content/drive
Resumed checkpoint with 59 completed replications.

SIMULATION 1 — MCMC-TUNED PILOT
---------------------------------------------
Spectral regimes: ['clear', 'weak']
Training sizes: [250, 500, 1000]
N_j: [8, 10, 12, 9]
K_j: [4, 4, 4, 4]
Candidate budgets: [39, 60, 80, 100, 120, 140, 156]
Minimum budget: 39
Maximum budget: 156
Budget tolerance zeta_n: 0.0015
Noise SD: 0.1
Monte Carlo replications per scenario: 100
Pilot mode: False
MCMC targets: theta = 0.25 , gamma = 0.25 , mu = 0.3
---------------------------------------------


[clear_n250] replication 60/100

Interrupted by user. Saving checkpoint...
[checkpoint] saved at 05:46:56


KeyboardInterrupt: 

In [2]:
# ============================================================
# SIMULATION 1 — ADDITIONAL n_train = 5000 EXPERIMENT
#
# Run this in a NEW CELL after the full Simulation 1 cell
# has already been executed.
#
# Everything is kept identical to the final rounded design.
# The ONLY statistical change is:
#
#     n_train = 5000
#
# Both spectral regimes are considered:
#     clear
#     weak
#
# 100 Monte Carlo replications per regime.
#
# Results and checkpoints are stored separately from the
# original n = 250, 500, 1000 experiment.
# ============================================================


# ============================================================
# 1. START / RESUME
# ============================================================

MODE_5000 = input(
    "Simulation 1 n=5000 extension: type 1 to START FROM THE BEGINNING "
    "or 2 to RESUME FROM THE LAST CHECKPOINT: "
).strip()

if MODE_5000 not in {"1", "2"}:

    raise ValueError(
        "Please run the cell again and enter either 1 or 2."
    )


# ============================================================
# 2. CHECK THAT THE MAIN SIMULATION CELL HAS BEEN RUN
# ============================================================

required_objects = [
    "run_replication",
    "build_true_parameters",
    "generate_dataset",
    "BUDGETS",
    "NOISE_SD",
    "BASE_SEED",
    "N_VAL",
    "N_TEST",
    "N_LEVELS",
    "K_LIST"
]

missing_objects = [
    x
    for x in required_objects
    if x not in globals()
]

if len(missing_objects) > 0:

    raise RuntimeError(
        "Run the complete Simulation 1 cell first. "
        "Missing objects: "
        + ", ".join(missing_objects)
    )


# ============================================================
# 3. n = 5000 CONFIGURATION
# ============================================================

N_TRAIN_5000 = 5000

N_REPS_5000 = 100

SCENARIOS_5000 = [

    {
        "name":
            "clear_n5000",

        "n_train":
            N_TRAIN_5000,

        "spectral_regime":
            "clear",

        "noise_sd":
            NOISE_SD
    },

    {
        "name":
            "weak_n5000",

        "n_train":
            N_TRAIN_5000,

        "spectral_regime":
            "weak",

        "noise_sd":
            NOISE_SD
    }
]


# ------------------------------------------------------------
# Use new scenario indices so that the random seeds do not
# overlap with the six original scenarios:
#
# 0 : clear 250
# 1 : clear 500
# 2 : clear 1000
# 3 : weak 250
# 4 : weak 500
# 5 : weak 1000
#
# New:
# 6 : clear 5000
# 7 : weak 5000
# ------------------------------------------------------------

SCENARIO_INDICES_5000 = {
    "clear":
        6,

    "weak":
        7
}


# ============================================================
# 4. SEPARATE OUTPUT DIRECTORY
# ============================================================

ROOT_5000 = os.path.join(
    ROOT,
    "n5000_extension"
)

os.makedirs(
    ROOT_5000,
    exist_ok=True
)


CHECKPOINT_PATH_5000 = os.path.join(
    ROOT_5000,
    "simulation1_n5000_checkpoint.pkl"
)

RESULTS_PATH_5000 = os.path.join(
    ROOT_5000,
    "simulation1_n5000_replication_results.csv"
)

BUDGET_RESULTS_PATH_5000 = os.path.join(
    ROOT_5000,
    "simulation1_n5000_budget_results.csv"
)

SUMMARY_PATH_5000 = os.path.join(
    ROOT_5000,
    "simulation1_n5000_summary.csv"
)

SE_PATH_5000 = os.path.join(
    ROOT_5000,
    "simulation1_n5000_standard_errors.csv"
)

COMBINED_SUMMARY_PATH = os.path.join(
    ROOT_5000,
    "simulation1_summary_with_n5000.csv"
)


# ============================================================
# 5. CONFIGURATION SIGNATURE
# ============================================================

CONFIG_5000 = {

    "experiment":
        "simulation1_n5000_extension",

    "n_train":
        N_TRAIN_5000,

    "n_reps":
        N_REPS_5000,

    "spectral_regimes":
        [
            "clear",
            "weak"
        ],

    "n_levels":
        N_LEVELS,

    "p_z":
        P_Z,

    "q":
        Q,

    "p_u":
        P_U,

    "n_val":
        N_VAL,

    "n_test":
        N_TEST,

    "noise_sd":
        NOISE_SD,

    "budgets":
        BUDGETS,

    "zeta_n":
        ZETA_N,

    "theta_lambda0":
        THETA_LAMBDA0,

    "tau_gamma":
        TAU_GAMMA,

    "tau_mu":
        TAU_MU,

    "sigma_a0":
        SIGMA_A0,

    "sigma_b0":
        SIGMA_B0,

    "red_hidden":
        RED_HIDDEN,

    "red_weight_decay":
        RED_WEIGHT_DECAY,

    "ref_epochs":
        REF_EPOCHS,

    "red_epochs":
        RED_EPOCHS,

    "ref_restarts":
        REF_RESTARTS,

    "red_restarts":
        RED_RESTARTS,

    "mcmc_iters":
        MCMC_ITERS,

    "mcmc_burn":
        MCMC_BURN,

    "mcmc_thin":
        MCMC_THIN,

    "adapt_every":
        MCMC_ADAPT_EVERY,

    "adapt_fraction":
        MCMC_ADAPT_FRACTION,

    "target_theta":
        MCMC_TARGET_THETA,

    "target_gamma":
        MCMC_TARGET_GAMMA,

    "target_mu":
        MCMC_TARGET_MU,

    "theta_initial_step":
        THETA_INITIAL_STEP,

    "gamma_initial_step":
        GAMMA_INITIAL_STEP,

    "mu_initial_step":
        MU_INITIAL_STEP,

    "mcmc_reference_n":
        MCMC_REFERENCE_N,

    "adapt_gain":
        MCMC_ADAPT_GAIN
}


CONFIG_SIGNATURE_5000 = hashlib.md5(

    json.dumps(
        CONFIG_5000,
        sort_keys=True
    ).encode()

).hexdigest()


# ============================================================
# 6. SAVE ORIGINAL GLOBAL STATE
#
# run_replication() uses the global save_checkpoint() function.
# We temporarily redirect checkpointing to a separate n=5000
# experiment and restore the original globals at the end.
# ============================================================

ORIGINAL_STATE = STATE

ORIGINAL_CHECKPOINT_PATH = CHECKPOINT_PATH

ORIGINAL_RESULTS_PATH = RESULTS_PATH

ORIGINAL_BUDGET_RESULTS_PATH = BUDGET_RESULTS_PATH

ORIGINAL_CHECKPOINT_CLOCK = CHECKPOINT_CLOCK

ORIGINAL_SAVE_CHECKPOINT = save_checkpoint

ORIGINAL_SAVE_TABLES = save_tables


# ============================================================
# 7. INITIALIZE / RESUME n = 5000 STATE
# ============================================================

if (
    MODE_5000 == "2"
    and
    os.path.exists(
        CHECKPOINT_PATH_5000
    )
):

    with open(
        CHECKPOINT_PATH_5000,
        "rb"
    ) as f:

        STATE_5000 = pickle.load(
            f
        )


    if (
        STATE_5000.get(
            "config_signature"
        )
        !=
        CONFIG_SIGNATURE_5000
    ):

        raise RuntimeError(
            "The n=5000 checkpoint was produced under a "
            "different configuration. Start again with option 1."
        )


    print(
        "Resumed n=5000 checkpoint with",
        len(
            STATE_5000[
                "completed"
            ]
        ),
        "completed replications."
    )

else:

    STATE_5000 = {

        "config_signature":
            CONFIG_SIGNATURE_5000,

        "completed":
            set(),

        "replication_rows":
            [],

        "budget_rows":
            [],

        "started":
            time.time()
    }


    if MODE_5000 == "2":

        print(
            "No compatible n=5000 checkpoint was found. "
            "Starting from the beginning."
        )


CHECKPOINT_CLOCK_5000 = {
    "last":
        time.time()
}


# ============================================================
# 8. TEMPORARILY REDIRECT GLOBAL CHECKPOINTING
# ============================================================

STATE = STATE_5000

CHECKPOINT_PATH = CHECKPOINT_PATH_5000

RESULTS_PATH = RESULTS_PATH_5000

BUDGET_RESULTS_PATH = BUDGET_RESULTS_PATH_5000

CHECKPOINT_CLOCK = CHECKPOINT_CLOCK_5000


def save_tables():

    if (
        len(
            STATE[
                "replication_rows"
            ]
        )
        >
        0
    ):

        pd.DataFrame(
            STATE[
                "replication_rows"
            ]
        ).to_csv(
            RESULTS_PATH,
            index=False
        )


    if (
        len(
            STATE[
                "budget_rows"
            ]
        )
        >
        0
    ):

        pd.DataFrame(
            STATE[
                "budget_rows"
            ]
        ).to_csv(
            BUDGET_RESULTS_PATH,
            index=False
        )


def save_checkpoint(
    force=False
):

    now = time.time()

    if (
        force
        or
        now
        -
        CHECKPOINT_CLOCK[
            "last"
        ]
        >=
        CHECKPOINT_SECONDS
    ):

        temporary_path = (
            CHECKPOINT_PATH
            +
            ".tmp"
        )


        with open(
            temporary_path,
            "wb"
        ) as f:

            pickle.dump(
                STATE,
                f,
                protocol=pickle.HIGHEST_PROTOCOL
            )


        os.replace(
            temporary_path,
            CHECKPOINT_PATH
        )


        save_tables()


        CHECKPOINT_CLOCK[
            "last"
        ] = now


        print(
            "[n=5000 checkpoint] saved at",
            time.strftime(
                "%H:%M:%S"
            )
        )


# ============================================================
# 9. RUN BOTH n = 5000 SCENARIOS
# ============================================================

print(
    "\n"
    "SIMULATION 1 — n_train = 5000 EXTENSION\n"
    "---------------------------------------------"
)

print(
    "Spectral regimes:",
    [
        "clear",
        "weak"
    ]
)

print(
    "Training sample size:",
    N_TRAIN_5000
)

print(
    "Replications per regime:",
    N_REPS_5000
)

print(
    "Candidate budgets:",
    BUDGETS
)

print(
    "Validation sample:",
    N_VAL,
    "| Test sample:",
    N_TEST
)

print(
    "---------------------------------------------\n"
)


try:

    for scenario in SCENARIOS_5000:

        scenario_index = (
            SCENARIO_INDICES_5000[
                scenario[
                    "spectral_regime"
                ]
            ]
        )


        for rep in range(
            1,
            N_REPS_5000 + 1
        ):

            task_key = (
                scenario[
                    "name"
                ],
                rep
            )


            if (
                task_key
                in
                STATE[
                    "completed"
                ]
            ):

                continue


            print(
                f"\n[{scenario['name']}] "
                f"replication {rep}/{N_REPS_5000}"
            )


            rep_row, budget_rows = run_replication(
                scenario,
                scenario_index,
                rep
            )


            STATE[
                "replication_rows"
            ].append(
                rep_row
            )


            STATE[
                "budget_rows"
            ].extend(
                budget_rows
            )


            STATE[
                "completed"
            ].add(
                task_key
            )


            save_checkpoint(
                force=True
            )


            print(
                "  B_hat =",
                rep_row[
                    "selected_budget"
                ],
                "| oracle B =",
                rep_row[
                    "test_oracle_budget"
                ],
                "| d =",
                rep_row[
                    "selected_d"
                ],
                "| recovery =",
                round(
                    rep_row[
                        "allocation_recovery_rate"
                    ],
                    3
                ),
                "| delta_norm =",
                round(
                    rep_row[
                        "normalized_delta_score_error"
                    ],
                    4
                ),
                "| tail RMSE =",
                round(
                    rep_row[
                        "posterior_tail_rmse"
                    ],
                    4
                ),
                "| test MSE =",
                round(
                    rep_row[
                        "final_test_mse"
                    ],
                    5
                )
            )


            print(
                "  post-burn acceptance:"
                " theta =",
                round(
                    rep_row[
                        "mean_theta_acceptance"
                    ],
                    3
                ),
                "| gamma =",
                round(
                    rep_row[
                        "gamma_acceptance"
                    ],
                    3
                ),
                "| mu =",
                round(
                    rep_row[
                        "mu_acceptance"
                    ],
                    3
                )
            )


except KeyboardInterrupt:

    print(
        "\nInterrupted by user. "
        "Saving n=5000 checkpoint..."
    )

    save_checkpoint(
        force=True
    )

    raise


except Exception:

    print(
        "\nAn error occurred. "
        "Saving n=5000 checkpoint..."
    )

    save_checkpoint(
        force=True
    )

    raise


finally:

    # --------------------------------------------------------
    # Keep a direct reference before restoring original globals
    # --------------------------------------------------------

    STATE_5000 = STATE


# ============================================================
# 10. SAVE FINAL n = 5000 RESULTS
# ============================================================

rep_5000_df = pd.DataFrame(
    STATE_5000[
        "replication_rows"
    ]
)

budget_5000_df = pd.DataFrame(
    STATE_5000[
        "budget_rows"
    ]
)


rep_5000_df.to_csv(
    RESULTS_PATH_5000,
    index=False
)

budget_5000_df.to_csv(
    BUDGET_RESULTS_PATH_5000,
    index=False
)


# ============================================================
# 11. n = 5000 SUMMARY
# ============================================================

summary_5000 = (

    rep_5000_df

    .groupby(
        [
            "spectral_regime",
            "n_train"
        ]
    )

    .agg(

        n_rep=(
            "rep",
            "count"
        ),

        allocation_recovery=(
            "allocation_recovery_rate",
            "mean"
        ),

        allocation_l1_error=(
            "allocation_mean_l1_error",
            "mean"
        ),

        structural_excess=(
            "mean_structural_excess",
            "mean"
        ),

        normalized_delta=(
            "normalized_delta_score_error",
            "mean"
        ),

        posterior_s2_rmse=(
            "posterior_s2_rmse",
            "mean"
        ),

        posterior_tail_rmse=(
            "posterior_tail_rmse",
            "mean"
        ),

        mean_selected_budget=(
            "selected_budget",
            "mean"
        ),

        mean_test_oracle_budget=(
            "test_oracle_budget",
            "mean"
        ),

        budget_match_rate=(
            "budget_exact_match",
            "mean"
        ),

        budget_absolute_error=(
            "budget_absolute_error",
            "mean"
        ),

        embedding_reduction=(
            "embedding_reduction",
            "mean"
        ),

        final_test_mse=(
            "final_test_mse",
            "mean"
        ),

        max_budget_test_mse=(
            "max_budget_test_mse",
            "mean"
        ),

        reference_test_mse=(
            "reference_test_mse",
            "mean"
        ),

        test_excess_vs_max_budget=(
            "test_excess_vs_max_budget",
            "mean"
        ),

        mean_theta_acceptance=(
            "mean_theta_acceptance",
            "mean"
        ),

        gamma_acceptance=(
            "gamma_acceptance",
            "mean"
        ),

        mu_acceptance=(
            "mu_acceptance",
            "mean"
        ),

        runtime_seconds=(
            "runtime_seconds",
            "mean"
        )
    )

    .reset_index()
)


summary_5000.to_csv(
    SUMMARY_PATH_5000,
    index=False
)


# ============================================================
# 12. MONTE CARLO STANDARD ERRORS
# ============================================================

def se_5000(
    x
):

    x = np.asarray(
        x,
        dtype=float
    )

    x = x[
        np.isfinite(
            x
        )
    ]


    if len(
        x
    ) <= 1:

        return np.nan


    return float(
        np.std(
            x,
            ddof=1
        )
        /
        np.sqrt(
            len(
                x
            )
        )
    )


se_rows_5000 = []


for (
    spectral_regime,
    n_train
), tmp in rep_5000_df.groupby(
    [
        "spectral_regime",
        "n_train"
    ]
):

    se_rows_5000.append(
        {

            "spectral_regime":
                spectral_regime,

            "n_train":
                n_train,

            "allocation_recovery_se":
                se_5000(
                    tmp[
                        "allocation_recovery_rate"
                    ]
                ),

            "normalized_delta_se":
                se_5000(
                    tmp[
                        "normalized_delta_score_error"
                    ]
                ),

            "posterior_tail_rmse_se":
                se_5000(
                    tmp[
                        "posterior_tail_rmse"
                    ]
                ),

            "selected_budget_se":
                se_5000(
                    tmp[
                        "selected_budget"
                    ]
                ),

            "embedding_reduction_se":
                se_5000(
                    tmp[
                        "embedding_reduction"
                    ]
                ),

            "final_test_mse_se":
                se_5000(
                    tmp[
                        "final_test_mse"
                    ]
                ),

            "test_excess_se":
                se_5000(
                    tmp[
                        "test_excess_vs_max_budget"
                    ]
                )
        }
    )


se_5000_df = pd.DataFrame(
    se_rows_5000
)

se_5000_df.to_csv(
    SE_PATH_5000,
    index=False
)


# ============================================================
# 13. COMBINE WITH ORIGINAL n = 250, 500, 1000 RESULTS
# ============================================================

try:

    original_rep_df = pd.DataFrame(
        ORIGINAL_STATE[
            "replication_rows"
        ]
    )


    combined_rep_df = pd.concat(
        [
            original_rep_df,
            rep_5000_df
        ],
        ignore_index=True
    )


    combined_summary = (

        combined_rep_df

        .groupby(
            [
                "spectral_regime",
                "n_train"
            ]
        )

        .agg(

            n_rep=(
                "rep",
                "count"
            ),

            allocation_recovery=(
                "allocation_recovery_rate",
                "mean"
            ),

            normalized_delta=(
                "normalized_delta_score_error",
                "mean"
            ),

            posterior_tail_rmse=(
                "posterior_tail_rmse",
                "mean"
            ),

            structural_excess=(
                "mean_structural_excess",
                "mean"
            ),

            mean_selected_budget=(
                "selected_budget",
                "mean"
            ),

            embedding_reduction=(
                "embedding_reduction",
                "mean"
            ),

            final_test_mse=(
                "final_test_mse",
                "mean"
            ),

            max_budget_test_mse=(
                "max_budget_test_mse",
                "mean"
            ),

            test_excess_vs_max_budget=(
                "test_excess_vs_max_budget",
                "mean"
            )
        )

        .reset_index()
    )


    combined_summary.to_csv(
        COMBINED_SUMMARY_PATH,
        index=False
    )


except Exception as e:

    combined_summary = None

    print(
        "Could not construct combined summary:",
        e
    )


# ============================================================
# 14. PRINT RESULTS
# ============================================================

print(
    "\n"
    "=============================================\n"
    "SIMULATION 1 — n_train = 5000 COMPLETED\n"
    "=============================================\n"
)

print(
    summary_5000.to_string(
        index=False
    )
)

print(
    "\nMonte Carlo standard errors:\n"
)

print(
    se_5000_df.to_string(
        index=False
    )
)


if combined_summary is not None:

    print(
        "\n"
        "Combined n = 250, 500, 1000, 5000 summary:\n"
    )

    print(
        combined_summary.to_string(
            index=False
        )
    )


# ============================================================
# 15. RESTORE ORIGINAL SIMULATION GLOBALS
# ============================================================

STATE = ORIGINAL_STATE

CHECKPOINT_PATH = ORIGINAL_CHECKPOINT_PATH

RESULTS_PATH = ORIGINAL_RESULTS_PATH

BUDGET_RESULTS_PATH = ORIGINAL_BUDGET_RESULTS_PATH

CHECKPOINT_CLOCK = ORIGINAL_CHECKPOINT_CLOCK

save_checkpoint = ORIGINAL_SAVE_CHECKPOINT

save_tables = ORIGINAL_SAVE_TABLES


print(
    "\nOriginal Simulation 1 globals restored."
)

print(
    "n=5000 results saved in:\n",
    ROOT_5000
)

Simulation 1 n=5000 extension: type 1 to START FROM THE BEGINNING or 2 to RESUME FROM THE LAST CHECKPOINT: 2
Resumed n=5000 checkpoint with 192 completed replications.

SIMULATION 1 — n_train = 5000 EXTENSION
---------------------------------------------
Spectral regimes: ['clear', 'weak']
Training sample size: 5000
Replications per regime: 100
Candidate budgets: [39, 60, 80, 100, 120, 140, 156]
Validation sample: 500 | Test sample: 3000
---------------------------------------------


[weak_n5000] replication 93/100
[n=5000 checkpoint] saved at 05:49:43
  B_hat = 156 | oracle B = 156 | d = (4, 4, 4, 4) | recovery = 1.0 | delta_norm = 0.0155 | tail RMSE = 0.0397 | test MSE = 0.0111
  post-burn acceptance: theta = 0.246 | gamma = 0.259 | mu = 0.284

[weak_n5000] replication 94/100
[n=5000 checkpoint] saved at 05:52:25
  B_hat = 156 | oracle B = 156 | d = (4, 4, 4, 4) | recovery = 1.0 | delta_norm = 0.0159 | tail RMSE = 0.0393 | test MSE = 0.01106
  post-burn acceptance: theta = 0.256 | g